## 📌 Introduction: What is the GPT API?

In this notebook, we use the OpenAI GPT API to automatically analyze text data.

The GPT API allows us to send a piece of text (called a *prompt*) to a large language model and receive structured outputs, such as:

- Sentiment classification (positive / negative / neutral)
- Short explanations (reasoning)
- Keyword extraction

Unlike traditional machine learning models, GPT does not require training on labeled data. Instead, it performs tasks based on instructions provided in the prompt.

This makes it especially useful for:
- Small datasets
- Exploratory analysis
- Rapid prototyping

In [1]:
# Install dependencies
from google.colab import files
import io
import json
import time
import pandas as pd
from openai import OpenAI

## ⚙️ How the GPT API Works

The core interaction with GPT happens through an API call:

1. We send a **prompt** (instruction + text data)
2. The model processes the request
3. The model returns a **response**

In this notebook, we use:

- `model="gpt-5.2"` → specifies which model to use  
- `input=prompt` → the instruction we send  

The model then generates an output based on the prompt.

Important:
- The quality of the output depends heavily on how the prompt is written
- This is known as *prompt engineering*

In [2]:
# 1. fill in the OpenAI api key
client = OpenAI(api_key="")# when using, please replace it with your own key

# 2. choose the column with main text
text_column = "text"


In [3]:
from datasets import load_dataset
import pandas as pd

# Load TweetEval sentiment dataset directly from Hugging Face
dataset = load_dataset("tweet_eval", "sentiment", split="test")

# Randomly sample 150 instances
sample = dataset.shuffle(seed=42).select(range(100))
few_shot_example = dataset.shuffle(seed=42).select(range(10))

# Convert to pandas DataFrame
df = pd.DataFrame(sample)
few_shot_df = pd.DataFrame(few_shot_example)

# Check result

pd.set_option('display.max_colwidth', None)
print(few_shot_df.head(10))
print(df.head(15))

README.md: 0.00B [00:00, ?B/s]

sentiment/train-00000-of-00001.parquet:   0%|          | 0.00/3.78M [00:00<?, ?B/s]

sentiment/test-00000-of-00001.parquet:   0%|          | 0.00/901k [00:00<?, ?B/s]

sentiment/validation-00000-of-00001.parq(…):   0%|          | 0.00/167k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

                                                                                                                                    text  \
0                                                                              @user @user @user @user but some of the reviews are good.   
1                                                                  US electoral college a rubber stamp, says Talbott #news #timesofindia   
2                     @user @user well traditionally we don't vote rightists into power. And are too fragmented to elect a trump or modi   
3            @user @user why did trump appoint Steve Bannon in his cabinet who openly racism and support KKK.This guy will save th wrld?   
4                                                                                AI and robots line up for battlefield service via @user   
5                                                                Galaxy Note 7 Banned In U.S. Flights & Violators Will Be Sent To Prison   
6                   

In [4]:
# Define label mapping
label_map = {
    0: "negative",
    1: "neutral",
    2: "positive"
}

# Create a new column with text labels
df["label_text"] = df["label"].map(label_map)

# Check result
pd.set_option('display.max_colwidth', None)
print(df.head(15))

                                                                                                                                     text  \
0                                                                               @user @user @user @user but some of the reviews are good.   
1                                                                   US electoral college a rubber stamp, says Talbott #news #timesofindia   
2                      @user @user well traditionally we don't vote rightists into power. And are too fragmented to elect a trump or modi   
3             @user @user why did trump appoint Steve Bannon in his cabinet who openly racism and support KKK.This guy will save th wrld?   
4                                                                                 AI and robots line up for battlefield service via @user   
5                                                                 Galaxy Note 7 Banned In U.S. Flights & Violators Will Be Sent To Prison   
6            

## 🧠 Prompt Design

The prompt is the most important part of using GPT.

In this notebook, the prompt includes:

### 1. Task definition
We clearly specify:
> "Classify the sentiment into positive, negative, or neutral"

### 2. Output constraints
We force the model to return:
- JSON format
- Exactly 3 fields: sentiment, reason, keywords

### 3. Instructions for ambiguity
We add:
> "If the sentiment is mixed, choose the dominant tone"

### 4. Context specification
We specify:
> "Tweet" (instead of news article)

This is important because:
- Social media text is short and informal
- The model behaves differently depending on context

👉 Well-designed prompts reduce errors and improve consistency.

## 📦 Parsing the Model Output

The model returns its response as plain text. However, we instruct the model to format its output as JSON.

For example, the model returns something like:

{
  "sentiment": "positive",
  "reason": "expresses excitement",
  "keywords": ["AI", "robots", "technology", "military", "future"]
}

We then convert this text into a Python dictionary using:

```python
result = json.loads(raw_output)

In [5]:
sentiment_list = []
reason_list = []
keywords_list = []

for text in df[text_column]:
    if pd.isna(text) or str(text).strip() == "":
        sentiment_list.append("ERROR")
        reason_list.append("Empty text")
        keywords_list.append(["", "", "", "", ""])
        continue

    article_text = str(text)[:12000]

    prompt = f"""
You are a sentiment classifier for social media posts.

Classify the sentiment of the given tweet into exactly one label: positive, negative, or neutral.

Ignore usernames (e.g., @user) and focus on the actual message content.

Return ONLY valid JSON.
Do not include any extra text, markdown, or explanations.

Use exactly this format:
{{
  "sentiment": "positive"
}}

If the sentiment is mixed, choose the dominant tone.

### Examples:

Tweet: "but some of the reviews are good."
Output:
{{"sentiment": "positive"}}

Tweet: "US electoral college a rubber stamp, says Talbott #news #timesofindia"
Output:
{{"sentiment": "neutral"}}

Tweet: "well traditionally we don't vote rightists into power. And are too fragmented to elect a trump or modi"
Output:
{{"sentiment": "negative"}}

Tweet: "why did trump appoint Steve Bannon in his cabinet who openly racism and support KKK. This guy will save the world?"
Output:
{{"sentiment": "negative"}}

Tweet: "AI and robots line up for battlefield service"
Output:
{{"sentiment": "neutral"}}

Tweet: "Galaxy Note 7 banned in U.S. flights and violators will be sent to prison"
Output:
{{"sentiment": "negative"}}

Tweet: "You can still sign up for the evening seminar on drone strikes in counter-terror wars"
Output:
{{"sentiment": "neutral"}}

Tweet: "Is Ukraine headed for another revolution?"
Output:
{{"sentiment": "neutral"}}

Tweet: "RIP EU Persona 5 PSN theme ;_; ;_; ;_; The ride never ends for Europe"
Output:
{{"sentiment": "neutral"}}

Tweet: "I bought a guy a Valentines Day gift once and that was really traumatic for me so I try my best to be single on gift giving holidays."
Output:
{{"sentiment": "negative"}}

### Now classify this:

Tweet:
\"\"\"
{article_text}
\"\"\"
"""

    try:
        response = client.responses.create(
            model="gpt-5.2",
            input=prompt
        )

        raw_output = response.output_text.strip()
        result = json.loads(raw_output)

        sentiment = str(result.get("sentiment", "")).lower().strip()
        reason = str(result.get("reason", "")).strip()
        keywords = result.get("keywords", [])

        if sentiment not in ["positive", "negative", "neutral"]:
            sentiment = "ERROR"
            reason = "Invalid label returned"

        if not isinstance(keywords, list):
            keywords = []

        keywords = [str(k).strip() for k in keywords[:5]]

        while len(keywords) < 5:
            keywords.append("")

    except Exception as e:
        sentiment = "ERROR"
        reason = str(e)
        keywords = ["", "", "", "", ""]

    sentiment_list.append(sentiment)
    reason_list.append(reason)
    keywords_list.append(", ".join(keywords))

    time.sleep(1)

# add results back to dataframe
df["GPT_sentiment"] = sentiment_list

# save result
df.to_csv("tweet_llm_labeled.csv", index=False, encoding="utf-8-sig")

# download
files.download("tweet_llm_labeled.csv")

print(df.head(10))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

                                                                                                                                    text  \
0                                                                              @user @user @user @user but some of the reviews are good.   
1                                                                  US electoral college a rubber stamp, says Talbott #news #timesofindia   
2                     @user @user well traditionally we don't vote rightists into power. And are too fragmented to elect a trump or modi   
3            @user @user why did trump appoint Steve Bannon in his cabinet who openly racism and support KKK.This guy will save th wrld?   
4                                                                                AI and robots line up for battlefield service via @user   
5                                                                Galaxy Note 7 Banned In U.S. Flights & Violators Will Be Sent To Prison   
6                   